In [1]:
# Cell 1: Cài đặt LeRobot & ffmpeg
!pip install --upgrade pip
!pip install "lerobot[training,dataset]>=0.6.0" accelerate torchvision
!apt-get update && apt-get install -y ffmpeg

# Kiểm tra nhận diện 2 GPU T4
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} - {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.7 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 91.2 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 88.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 109.3 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: packaging
    Found existing installation: packaging 26.3
    Uninstalling packaging-26.3:
      Successfully uninstalled packaging-26.3
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.14.0.94
    Uninstalling opencv-python-headless-4.14.0.94:━━━━━━━━━━━━━━━━  2/11 [opencv-python-headless]
      Successfully uninstalled opencv-python-headless-4.14.0

In [3]:
# Cell 2: Xác thực Hugging Face
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ Đã đăng nhập Hugging Face thành công từ Kaggle Secret!")
except Exception as e:
    # Nếu không dùng secret, bạn có thể nhập trực tiếp token ở đây:
    hf_token = "YOUR_HF_TOKEN_HERE"
    login(token=hf_token)

✅ Đã đăng nhập Hugging Face thành công từ Kaggle Secret!


In [4]:
import json
from pathlib import Path
from huggingface_hub import snapshot_download

# 0. Cài đặt dependency bắt buộc cho SmolVLM processor
!pip install -q num2words

# 1. Tạo package plugin chuẩn của LeRobot (tiền tố 'lerobot_policy_' để lerobot-train tự động import trên mọi GPU)
plugin_root = Path("/kaggle/working/lerobot_policy_chess_plugins")
pkg_dir = plugin_root / "lerobot_policy_chess_plugins"
pkg_dir.mkdir(parents=True, exist_ok=True)

(plugin_root / "pyproject.toml").write_text("""[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"

[project]
name = "lerobot_policy_chess_plugins"
version = "0.1.0"
dependencies = []
""")

(pkg_dir / "__init__.py").write_text("""from dataclasses import dataclass
from typing import Any
import torch

from lerobot.configs import PipelineFeatureType, PolicyFeature
from lerobot.lerobot_types import TransitionKey
from lerobot.processor import ObservationProcessorStep, ProcessorStepRegistry
from lerobot.utils.constants import OBS_STATE


@dataclass
@ProcessorStepRegistry.register("chess_goal_noise")
class ChessGoalNoiseProcessorStep(ObservationProcessorStep):
    \"\"\"Injects physical Gaussian noise (in meters) into ee_xyz [6:9] and delta_xyz [9:12]
    of the 12D observation.state strictly during policy training.
    \"\"\"
    goal_noise_std: float = 0.008
    ee_noise_std: float = 0.003
    max_noise_clip: float = 0.022
    noise_prob: float = 0.8
    enabled: bool = True

    def get_config(self) -> dict[str, Any]:
        return {
            "goal_noise_std": self.goal_noise_std,
            "ee_noise_std": self.ee_noise_std,
            "max_noise_clip": self.max_noise_clip,
            "noise_prob": self.noise_prob,
            "enabled": self.enabled,
        }

    def observation(self, observation: dict[str, Any]) -> dict[str, Any]:
        if not self.enabled:
            return observation
        # Chỉ bơm nhiễu trong pha Training (có gradient và có nhãn action)
        if not torch.is_grad_enabled() or self.transition.get(TransitionKey.ACTION) is None:
            return observation
        if OBS_STATE not in observation:
            return observation

        state = observation[OBS_STATE]
        if not isinstance(state, torch.Tensor) or state.shape[-1] < 12:
            return observation

        state = state.clone()
        bsize = state.shape[0]
        mask = (torch.rand(bsize, 1, device=state.device) < self.noise_prob).to(state.dtype)
        if state.ndim == 3:
            mask = mask.unsqueeze(1)

        # 1. Nhiễu vị trí TCP ee_xyz (indices 6:9) và bù trừ tương ứng vào delta_xyz
        if self.ee_noise_std > 0:
            ee_noise = torch.clamp(
                torch.randn_like(state[..., 6:9]) * self.ee_noise_std,
                -self.max_noise_clip,
                self.max_noise_clip,
            )
            state[..., 6:9] = state[..., 6:9] + ee_noise * mask
            state[..., 9:12] = state[..., 9:12] - ee_noise * mask

        # 2. Nhiễu mục tiêu delta_xyz = P_goal - P_ee (indices 9:12)
        if self.goal_noise_std > 0:
            goal_noise = torch.clamp(
                torch.randn_like(state[..., 9:12]) * self.goal_noise_std,
                -self.max_noise_clip,
                self.max_noise_clip,
            )
            state[..., 9:12] = state[..., 9:12] + goal_noise * mask

        observation[OBS_STATE] = state
        return observation

    def transform_features(
        self, features: dict[PipelineFeatureType, dict[str, PolicyFeature]]
    ) -> dict[PipelineFeatureType, dict[str, PolicyFeature]]:
        return features
""")

# Cài đặt editable package để cả 2 tiến trình GPU đều tự động nạp qua register_third_party_plugins()
!pip install -q -e /kaggle/working/lerobot_policy_chess_plugins --no-build-isolation --no-deps

# 2. Tải base model lerobot/smolvla_base và bơm chess_goal_noise vào policy_preprocessor.json
patched_model_dir = Path("/kaggle/working/smolvla_base_patched")
snapshot_download("lerobot/smolvla_base", local_dir=str(patched_model_dir))

# Xóa cứng input_features/output_features cũ của smolvla_base để make_policy tự động nhận diện
# đúng 3 camera (wrist, side, overhead), state 12D và action 6D từ DuyKhongCay/lekiwi_chess_v7
cfg_path = patched_model_dir / "config.json"
cfg_data = json.loads(cfg_path.read_text())
cfg_data["input_features"] = {}
cfg_data["output_features"] = {}
cfg_data["empty_cameras"] = 0
cfg_path.write_text(json.dumps(cfg_data, indent=2))

# Chèn bước chess_goal_noise ngay trước normalizer_processor trong policy_preprocessor.json
pre_path = patched_model_dir / "policy_preprocessor.json"
pre_data = json.loads(pre_path.read_text())
pre_data["steps"] = [s for s in pre_data["steps"] if s.get("registry_name") != "chess_goal_noise"]
norm_idx = next(
    i for i, s in enumerate(pre_data["steps"]) if s.get("registry_name") == "normalizer_processor"
)
pre_data["steps"].insert(
    norm_idx,
    {
        "registry_name": "chess_goal_noise",
        "config": {
            "goal_noise_std": 0.008,
            "ee_noise_std": 0.003,
            "max_noise_clip": 0.022,
            "noise_prob": 0.8,
            "enabled": True,
        },
    },
)
pre_path.write_text(json.dumps(pre_data, indent=2))
print("Patched policy_preprocessor.json steps:", [s.get("registry_name") for s in pre_data["steps"]])

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lerobot_policy_chess_plugins (pyproject.toml) ... done


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Patched policy_preprocessor.json steps: ['rename_observations_processor', 'to_batch_processor', 'smolvla_new_line_processor', 'tokenizer_processor', 'device_processor', 'chess_goal_noise', 'normalizer_processor']


In [ ]:
import os
import yaml

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 1. Xóa sạch cache LeRobot Hub và output cũ
!rm -rf /root/.cache/huggingface/lerobot/hub/datasets--DuyKhongCay--lekiwi_chess_v7
!rm -rf /root/.cache/huggingface/hub/datasets--DuyKhongCay--lekiwi_chess_v7
!rm -rf /root/.cache/huggingface/lerobot/DuyKhongCay/lekiwi_chess_v7
!rm -rf /kaggle/working/outputs/act_lekiwi_chess
!rm -rf /kaggle/working/outputs/lekiwi_chess_smolvla

# 2. Tạo file config YAML chuẩn hóa cho 2x T4 GPUs (Effective Batch Size = 16 x 2 = 32, ~8.5h)
config = {
    "dataset": {
        "repo_id": "DuyKhongCay/lekiwi_chess_v7",
        "revision": "main",
        "return_uint8": True,
        "video_backend": "pyav",
    },
    "policy": {
        "path": "/kaggle/working/smolvla_base_patched",
        "push_to_hub": True,
        "repo_id": "DuyKhongCay/lekiwi_chess_smolvla",
        "private": False,
        "use_amp": False,
        "freeze_vision_encoder": True,
        "train_expert_only": True,
        "train_state_proj": True,
        # 1. Áp dụng khuyến nghị Action Insights từ Hugging Face
        "chunk_size": 30,  # Giảm từ 50 xuống 30 (1.20s): vừa khớp nhịp tay robot, tăng độ chính xác gắp cờ
        "n_action_steps": 30,  # Luôn <= chunk_size
        # Giữ nguyên độ phân giải gốc 384x384 (thay vì 512x512) để giảm 50% visual tokens
        "resize_imgs_with_padding": [384, 384],
        "optimizer_lr": 1e-4,
        "scheduler_warmup_steps": 1000,
        "scheduler_decay_steps": 35000,
        "scheduler_decay_lr": 2.5e-6,
    },
    "output_dir": "/kaggle/working/outputs/lekiwi_chess_smolvla",
    "job_name": "lekiwi_chess_smolvla_v7_#1",
    "steps": 13000,
    "batch_size": 8,
    "num_workers": 2,
    "prefetch_factor": 2,
    "persistent_workers": True,
    "log_freq": 100,
    "save_checkpoint": True,
    "save_freq": 2500,
    "save_checkpoint_to_hub": True,
    "seed": 1000,
}

with open("/kaggle/working/train_config.yaml", "w") as f:
    yaml.dump(config, f, sort_keys=False)

# 3. Chạy huấn luyện song song trên 2x GPU T4 qua --config_path
!accelerate launch \
    --multi_gpu \
    --num_processes=2 \
    --num_machines=1 \
    --mixed_precision=no \
    --dynamo_backend=no \
    $(which lerobot-train) --config_path="/kaggle/working/train_config.yaml"

INFO 2026-10-29 15:19:54 ot_train.py:272 {'batch_size': 8,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataloader_multiprocessing_context': 'spawn',
 'dataset': {'depth_output_unit': 'mm',
             'episodes': None,
             'eval_split': 0.0,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8